# 🧠 Visualisasi Partial Directed Coherence (PDC) EEG - 2D Top-Down (Semua Frekuensi)

Notebook ini digunakan untuk memvisualisasikan matriks **Partial Directed Coherence (PDC)** dalam **proyeksi 2D arah atas (top-down/axial view)** secara otomatis untuk seluruh subjek, sesi, dan pita frekuensi.

### Sumber Data Baru:
Notebook ini langsung membaca berkas PDC per trial dari direktori data baru (`new_data/02_pdc/output/pdc_matrices/`), kemudian **merata-rata trial berdasarkan kondisi emosi** (Negative, Neutral, Positive) menggunakan label emosi SEED standar sebelum melakukan pemetaan 2D.

### Fitur Utama:
1. **Dukungan 6 Pita Frekuensi:** Memvisualisasikan **Delta, Theta, Alpha, Beta, Gamma, dan Broadband**.
2. **Aglomerasi Otomatis:** Mengelompokkan dan merata-rata trial berdasarkan emosi:
   * **Positive (Emosi Positif):** Trial 1, 6, 9, 10, 14
   * **Neutral (Emosi Netral):** Trial 2, 5, 8, 11, 13
   * **Negative (Emosi Negatif):** Trial 3, 4, 7, 12, 15
3. **Penyimpanan Terstruktur:** Hasil visualisasi disimpan rapi pada subfolder per subjek dan sesi (contoh: `figures/subject_01/session_20131027/`).

In [5]:
# ==============================================================================
# SECTION 1: IMPORT LIBRARIES
# ==============================================================================
import os
import warnings
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from tqdm import tqdm

# Abaikan warning matplotlib
warnings.filterwarnings('ignore')

## 📍 Pemetaan Koordinat Elektroda 2D (SEED 62 Channel)

Memetakan elektroda SEED ke koordinat 2D berdasarkan template 3D standard 10-05 MNE-Python.

In [6]:
# ==============================================================================
# SECTION 2: CHANNEL CONFIGURATION & 2D COORDINATES
# ==============================================================================

# Nama 62 elektroda EEG standar dari dataset SEED
channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'AF3', 'AF4', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8',
    'FT7', 'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'FT8', 'T7', 'C5', 'C3', 'C1',
    'Cz', 'C2', 'C4', 'C6', 'T8', 'TP7', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6',
    'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO5', 'PO3',
    'POz', 'PO4', 'PO6', 'PO8', 'CB1', 'O1', 'Oz', 'O2', 'CB2'
]

# Pembagian lobus otak untuk pewarnaan node
frontal_lobes = ['FP1', 'FPZ', 'FP2', 'AF3', 'AF4', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8']
central_lobes = ['FT7', 'FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'FT8', 'T7', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'T8']
parietal_lobes = ['TP7', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8']
occipital_lobes = ['PO7', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'PO8', 'CB1', 'O1', 'OZ', 'O2', 'CB2']

node_colors = []
for name in channel_names:
    ch_upper = name.upper()
    if ch_upper in frontal_lobes:
        node_colors.append('#E84A5F')  # Coral Red
    elif ch_upper in central_lobes:
        node_colors.append('#3EC1D3')  # Cyan
    elif ch_upper in parietal_lobes:
        node_colors.append('#F9A826')  # Golden Yellow
    elif ch_upper in occipital_lobes:
        node_colors.append('#4A47A3')  # Indigo Purple
    else:
        node_colors.append('#95A5A6')

# Ambil posisi elektroda dari database MNE
montage_1005 = mne.channels.make_standard_montage('standard_1005')
positions_db = montage_1005.get_positions()['ch_pos']

node_coords_2d = {}
for name in channel_names:
    ch_upper = name.upper()
    if ch_upper == 'CB1':
        pos = positions_db['I1']
    elif ch_upper == 'CB2':
        pos = positions_db['I2']
    else:
        pos = None
        for k in positions_db.keys():
            if k.upper() == ch_upper:
                pos = positions_db[k]
                break
        if pos is None:
            pos = np.array([0.0, 0.0, 0.0])
    node_coords_2d[name] = (pos[0] * 12, pos[1] * 12)

print("✓ Berhasil memetakan koordinat 2D elektroda.")

✓ Berhasil memetakan koordinat 2D elektroda.


## 📐 Fungsi Utama Visualisasi 2D Top-Down

Fungsi `plot_eeg_connectome_2d` menggambar outline kepala standar (lingkaran kepala, hidung, telinga), memproses matriks PDC, menyaring koneksi terkuat, dan menggambar panah melengkung.

In [7]:
# ==============================================================================
# SECTION 3: EEG 2D TOP-DOWN CONNECTOME PLOTTING FUNCTION
# ==============================================================================

def plot_eeg_connectome_2d(pdc_matrix, ax, title, n_lines=35):
    """
    Memplot konektivitas EEG 2D dari atas pada axis Matplotlib berdasarkan nilai PDC.
    """
    head_radius = 1.0
    
    # 1. Gambar outline kepala
    head_circle = plt.Circle((0, 0), head_radius, fill=False, color='#7F8C8D', linewidth=2.0)
    ax.add_patch(head_circle)
    
    # Hidung (nose)
    nose_x = [-0.10, 0, 0.10]
    nose_y = [head_radius, head_radius + 0.12, head_radius]
    ax.plot(nose_x, nose_y, color='#7F8C8D', linewidth=2.0)
    
    # Telinga (ears)
    left_ear = mpatches.Ellipse((-head_radius - 0.05, 0), 0.06, 0.20, fill=False, color='#7F8C8D', linewidth=1.5)
    ax.add_patch(left_ear)
    right_ear = mpatches.Ellipse((head_radius + 0.05, 0), 0.06, 0.20, fill=False, color='#7F8C8D', linewidth=1.5)
    ax.add_patch(right_ear)
    
    # 2. Filter data konektivitas
    pdc_work = pdc_matrix.copy()
    np.fill_diagonal(pdc_work, 0)  # Hilangkan self-loops
    
    if np.any(pdc_work > 0):
        total_valid = np.count_nonzero(pdc_work)
        if total_valid > n_lines:
            threshold_val = np.percentile(pdc_work[pdc_work > 0], 100 - (100 * n_lines / total_valid))
        else:
            threshold_val = pdc_work[pdc_work > 0].min()
    else:
        threshold_val = 0.1
        
    adj_matrix = np.where(pdc_work >= threshold_val, pdc_work, 0)
    
    # Derajat node untuk penskalaan ukuran
    degrees = np.sum(adj_matrix > 0, axis=0) + np.sum(adj_matrix > 0, axis=1)
    deg_min, deg_max = degrees.min(), degrees.max()
    node_sizes = 50 + 130 * (degrees - deg_min) / (deg_max - deg_min + 1e-5)
    
    # Kumpulkan koneksi terkuat
    connections = []
    n_nodes = len(channel_names)
    for i in range(n_nodes):
        for j in range(n_nodes):
            if adj_matrix[i, j] > 0:
                connections.append({
                    'from': channel_names[i],
                    'to': channel_names[j],
                    'strength': adj_matrix[i, j]
                })
    connections.sort(key=lambda x: x['strength'])
    
    if not connections:
        ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
        return ax
        
    # Konfigurasi cmap (biru untuk membedakan dari Granger Causality)
    cmap = plt.colormaps.get_cmap('Blues')
    strengths = [c['strength'] for c in connections]
    norm = Normalize(vmin=min(strengths), vmax=max(strengths))
    
    # 3. Plot panah koneksi (melengkung)
    for conn in connections:
        from_ch, to_ch = conn['from'], conn['to']
        strength = conn['strength']
        
        x1, y1 = node_coords_2d[from_ch]
        x2, y2 = node_coords_2d[to_ch]
        
        color = cmap(norm(strength))
        linewidth = 1.0 + 3.5 * norm(strength)
        alpha = 0.70 + 0.30 * norm(strength)
        
        dx, dy = x2 - x1, y2 - y1
        dist = np.sqrt(dx**2 + dy**2)
        if dist > 0:
            shrink = 0.08
            x1_new = x1 + shrink * dx / dist
            y1_new = y1 + shrink * dy / dist
            x2_new = x2 - shrink * dx / dist
            y2_new = y2 - shrink * dy / dist
            
            ax.annotate('', 
                        xy=(x2_new, y2_new), 
                        xytext=(x1_new, y1_new),
                        arrowprops=dict( 
                            arrowstyle='-|>',
                            color=color,
                            alpha=alpha,
                            lw=linewidth,
                            mutation_scale=12,
                            connectionstyle='arc3,rad=0.15'
                        ))
            
    # 4. Plot bulatan elektroda & label teks bersih
    for idx, name in enumerate(channel_names):
        x, y = node_coords_2d[name]
        color = node_colors[idx]
        size = node_sizes[idx]
        
        ax.scatter(x, y, s=size, color=color, zorder=10, edgecolors='white', linewidths=1.2, alpha=0.95)
        ax.text(x, y + 0.04, name, ha='center', va='bottom', fontsize=8, fontweight='bold', color='#2C3E50', zorder=12)
        
    ax.set_xlim(-head_radius - 0.2, head_radius + 0.2)
    ax.set_ylim(-head_radius - 0.2, head_radius + 0.2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold', color='#1A252C', pad=15)

## 🚀 Loop Pemrosesan Otomatis - Semua Subjek, Sesi, dan Frekuensi Band

Langkah ini memindai folder data input baru, mendeteksi seluruh subjek (`subject_01` hingga `subject_15`), memproses seluruh sesi aktif, membagi trial menjadi 3 emosi, merata-rata nilainya, dan menyimpan visualisasi 2D ke:

`new_data/02_pdc/figures/<subject_id>/<session_date>/`

In [8]:
# ==============================================================================
# SECTION 4: LOOP EXECUTION FOR ALL SUBJECTS & ALL SESSIONS (ALL BANDS)
# ==============================================================================

base_dir = r"d:\Skripsi"
pdc_matrices_dir = os.path.join(base_dir, "new_data", "02_pdc", "output", "pdc_matrices")
output_base_dir = os.path.join(base_dir, "new_data", "02_pdc", "figures")

# Definisi Trial Labels SEED (1 = positive, 0 = neutral, -1 = negative)
TRIAL_LABELS = [1, 0, -1, -1, 0, 1, -1, 0, 1, 1, 0, -1, 0, 1, -1]
emotion_trials = {
    "positive": [idx + 1 for idx, label in enumerate(TRIAL_LABELS) if label == 1],
    "neutral": [idx + 1 for idx, label in enumerate(TRIAL_LABELS) if label == 0],
    "negative": [idx + 1 for idx, label in enumerate(TRIAL_LABELS) if label == -1]
}

# Cari semua folder subjek
subjects = sorted([d for d in os.listdir(pdc_matrices_dir) if os.path.isdir(os.path.join(pdc_matrices_dir, d)) and d.startswith("subject_")])
print(f"Menemukan {len(subjects)} folder subjek untuk diproses.\n")

bands = ["delta", "theta", "alpha", "beta", "gamma", "broadband"]
emotions = ["negative", "neutral", "positive"]
processed_count = 0

# Loop subjek
for subject in tqdm(subjects, desc="Progress Subjek"):
    sub_path = os.path.join(pdc_matrices_dir, subject)
    sessions = sorted([d for d in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, d)) and d.startswith("session_")])
    
    for session in sessions:
        session_dir = os.path.join(sub_path, session)
        output_session_dir = os.path.join(output_base_dir, subject, session)
        
        # Loop pita frekuensi
        for band in bands:
            # Periksa apakah minimal ada satu file trial untuk band ini
            trial_one_path = os.path.join(session_dir, f"pdc_{band}_trial_01.npy")
            if not os.path.exists(trial_one_path):
                continue
                
            # Buat folder output sesi
            os.makedirs(output_session_dir, exist_ok=True)
            
            # Simpan matriks rata-rata per emosi
            emotion_matrices = {}
            for emotion in emotions:
                trial_indices = emotion_trials[emotion]
                matrices = []
                for trial_num in trial_indices:
                    file_path = os.path.join(session_dir, f"pdc_{band}_trial_{trial_num:02d}.npy")
                    if os.path.exists(file_path):
                        matrices.append(np.load(file_path))
                if len(matrices) > 0:
                    emotion_matrices[emotion] = np.mean(matrices, axis=0)
                    
            # Pastikan ketiga emosi memiliki data
            if len(emotion_matrices) < 3:
                continue
                
            # 1. Plot Perbandingan Side-by-Side (3 Emosi)
            fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(22, 8), facecolor='white')
            
            for idx, emotion in enumerate(emotions):
                plot_eeg_connectome_2d(emotion_matrices[emotion], axes[idx], f"{emotion.upper()} Emotion", n_lines=35)
                
            plt.suptitle(f"EEG PDC Connectivity Maps (Top View) - {band.upper()} Band\n{subject.replace('_', ' ').title()} | {session.replace('_', ' ').title()}", 
                         fontsize=20, fontweight='bold', color='#1A252C', y=1.02)
            
            save_path_comp = os.path.join(output_session_dir, f"pdc_2d_topdown_comparison_{band}.png")
            plt.savefig(save_path_comp, dpi=300, bbox_inches='tight')
            plt.close()
            
            # 2. Plot dan Simpan Gambar Individual
            for emotion in emotions:
                fig, ax = plt.subplots(figsize=(8, 8), facecolor='white')
                title_single = f"PDC {band.upper()} - {emotion.upper()}\n{subject.replace('_', ' ').title()} | {session.replace('_', ' ').title()}"
                plot_eeg_connectome_2d(emotion_matrices[emotion], ax, title_single, n_lines=35)
                
                save_path_single = os.path.join(output_session_dir, f"pdc_2d_topdown_{band}_{emotion}.png")
                plt.savefig(save_path_single, dpi=300, bbox_inches='tight')
                plt.close()
                
        processed_count += 1

print(f"\n✓ Selesai memproses total {processed_count} sesi untuk {len(subjects)} subjek.")
print(f"✓ Seluruh hasil gambar disimpan secara terstruktur di: {output_base_dir}")

Menemukan 15 folder subjek untuk diproses.



Progress Subjek: 100%|██████████| 15/15 [50:12<00:00, 200.84s/it]


✓ Selesai memproses total 45 sesi untuk 15 subjek.
✓ Seluruh hasil gambar disimpan secara terstruktur di: d:\Skripsi\new_data\02_pdc\figures
